# Task 2.1 Data Modeling with ORM (PonyORM)
## Oscar Explorer

This notebook loads **The Oscar Award** dataset into a SQLite database using **PonyORM**,
defines a normalised entity model, and verifies the schema with sample queries.

**Dataset:** `the_oscar_award.csv` has 11,110 nomination records across 97 ceremonies (1927â€“2024)  
**Columns:** `year_film`, `year_ceremony`, `ceremony`, `category`, `canon_category`, `name`, `film`, `winner`

In [ ]:
import os
import pandas as pd
from pony.orm import (
    Database, PrimaryKey, Required, Optional, Set, composite_key,
    db_session, select, count, desc, commit
)

In [ ]:
#  Paths 
BASE_DIR = os.path.dirname(os.path.abspath("task2_oscar.ipynb"))
CSV_PATH = os.path.join(BASE_DIR, "The Oscar Award", "the_oscar_award.csv")
DB_PATH  = os.path.join(BASE_DIR, "oscar.db")

# Start fresh so the notebook is fully reproducible
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
    print(f"Removed existing oscar.db")

print(f"CSV exists: {os.path.exists(CSV_PATH)}")
print(f"DB will be created at: {DB_PATH}")

Removed existing oscar.db
CSV exists: True
DB will be created at: c:\Users\user\OneDrive\PycharmProjects\Big Data scraping\task2\oscar.db


## Schema Design

The raw CSV is a **flat nomination log** where every row is one person/film nominated in one category at one ceremony.
Storing it flat would repeat ceremony years, category names, and person names thousands of times.
A normalised relational schema avoids this redundancy and enables efficient ORM queries.

### Entities

| Entity | PK | Key fields | Rationale |
|---|---|---|---|
| `Ceremony` | `number` (int) | `year` | 97 distinct ceremonies; number is the natural identifier |
| `Category` | auto int | `name` (unique), `canon_name` | 118 specific categories that collapse into 58 canonical ones |
| `Person` | auto int | `name` (unique) | 6,893 distinct nominees; deduplicated to avoid per-row repetition |
| `Film` | auto int | `title` + `year` (composite unique) | 5,233 unique (title, year_film) pairs; a film can earn multiple nominations |
| `Nomination` | auto int | FK all four above + `winner` | Junction table has each row is one nomination record |

### Nullable relationships
- `Nomination.person` is **Optional**  7 rows have no name (e.g. honorary foreign-film awards where the film itself is the awardee).
- `Nomination.film` is **Optional** 359 early rows have no film attribution (sound studios, title writers, etc.).

### Entity-Relationship diagram (text)
```
Ceremony ──< Nomination >── Category
                │
              Person (optional)
                │
               Film (optional)

```

In [ ]:
#  PonyORM entity definitions 
db = Database()


class Ceremony(db.Entity):
    number = PrimaryKey(int)          # e.g. 1, 2, ¦ 97
    year   = Required(int)            # year the ceremony was held
    nominations = Set('Nomination')


class Category(db.Entity):
    name       = Required(str, unique=True)   # e.g. "ACTOR IN A LEADING ROLE"
    canon_name = Required(str)                # e.g. "ACTOR"
    nominations = Set('Nomination')


class Person(db.Entity):
    name        = Required(str, unique=True)
    nominations = Set('Nomination')


class Film(db.Entity):
    title       = Required(str)
    year        = Required(int)               # year_film (release year)
    nominations = Set('Nomination')
    # Composite unique: same title can exist in different years (remakes)
    composite_key(title, year)


class Nomination(db.Entity):
    ceremony = Required(Ceremony)
    category = Required(Category)
    person   = Optional(Person)    # null for awards with no named individual
    film     = Optional(Film)      # null for early awards with no film attribution
    winner   = Required(bool)


# Bind to SQLite and create all tables
db.bind(provider='sqlite', filename=DB_PATH, create_db=True)
db.generate_mapping(create_tables=True)
print("Schema created 5 tables:", [e.__name__ for e in db.entities.values()])

Schema created 5 tables: ['Ceremony', 'Category', 'Person', 'Film', 'Nomination']


## Why PonyORM?

The task allowed PonyORM, SQLAlchemy, or Peewee. PonyORM was chosen for three reasons:

| Criterion | PonyORM | SQLAlchemy | Peewee |
|---|---|---|---|
| **Query syntax** | Native Python generator expressions (`select(n for n in Nomination if n.winner)`) has no string SQL, no chained `.filter()` calls | Verbose; requires `session`, `query()`, explicit join declarations | Simple but less expressive for complex joins |
| **Schema definition** | Declarative entity classes with automatic relationship inference from `Set` / `Required` / `Optional` | Explicit `relationship()` + `ForeignKey()` declarations on both sides | Similar to PonyORM but with less automatic reverse-relation handling |
| **IDE-friendly** | Entities are plain Python classes; attributes are accessible with dot notation and full autocomplete | Core layer uses string column names; ORM layer is better but more boilerplate | Good, but less mature type-hint support |

The main trade-off is that PonyORM is less widely deployed in production than SQLAlchemy,
but for an exploratory data notebook its concise generator syntax makes complex queries
significantly more readable.

In [ ]:
#  Load CSV and populate ORM entities 
df = pd.read_csv(CSV_PATH)
print(f"CSV loaded: {len(df):,} rows")

@db_session
def populate():
    # Ceremonies
    for _, row in df[['ceremony', 'year_ceremony']].drop_duplicates().iterrows():
        Ceremony(number=int(row['ceremony']), year=int(row['year_ceremony']))
    commit()
    print(f"Ceremonies inserted: {db.execute('SELECT COUNT(*) FROM Ceremony').fetchone()[0]}")

    # Categories
    for _, row in df[['category', 'canon_category']].drop_duplicates().iterrows():
        Category(name=str(row['category']), canon_name=str(row['canon_category']))
    commit()
    print(f"Categories inserted: {db.execute('SELECT COUNT(*) FROM Category').fetchone()[0]}")

    # Persons
    for name in df['name'].dropna().unique():
        Person(name=str(name))
    commit()
    print(f"Persons inserted: {db.execute('SELECT COUNT(*) FROM Person').fetchone()[0]}")

    # Films 
    film_pairs = (df[['film', 'year_film']]
                  .dropna(subset=['film'])
                  .drop_duplicates())
    for _, row in film_pairs.iterrows():
        Film(title=str(row['film']), year=int(row['year_film']))
    commit()
    print(f"Films inserted: {db.execute('SELECT COUNT(*) FROM Film').fetchone()[0]}")

    # Nominations 
    # Build lookup caches to avoid repeated DB queries inside the loop
    ceremonies  = {c.number: c for c in Ceremony.select()}
    categories  = {c.name:   c for c in Category.select()}
    persons     = {p.name:   p for p in Person.select()}
    films       = {(f.title, f.year): f for f in Film.select()}

    for _, row in df.iterrows():
        person = persons.get(str(row['name'])) if pd.notna(row['name']) else None
        film   = films.get((str(row['film']), int(row['year_film']))) if pd.notna(row['film']) else None
        Nomination(
            ceremony = ceremonies[int(row['ceremony'])],
            category = categories[str(row['category'])],
            person   = person,
            film     = film,
            winner   = bool(row['winner']),
        )
    commit()
    print(f"Nominations inserted: {db.execute('SELECT COUNT(*) FROM Nomination').fetchone()[0]}")

populate()
print("\nAll entities loaded successfully.")

CSV loaded: 11,110 rows
Ceremonies inserted: 97
Categories inserted: 118
Persons inserted: 6893
Films inserted: 5233
Nominations inserted: 11110

All entities loaded successfully.


## Verification

In [ ]:
#  Row counts 
@db_session
def row_counts():
    counts = {
        'Ceremony':   count(c for c in Ceremony),
        'Category':   count(c for c in Category),
        'Person':     count(p for p in Person),
        'Film':       count(f for f in Film),
        'Nomination': count(n for n in Nomination),
    }
    display(pd.DataFrame(list(counts.items()), columns=['Table', 'Rows']))

row_counts()

,Table,Rows
0,Ceremony,97
1,Category,118
2,Person,6893
3,Film,5233
4,Nomination,11110


In [ ]:
#  Sample ORM query 1: top 10 most-nominated persons 
@db_session
def top_nominees():
    results = select(
        (p.name, count(n))
        for p in Person
        for n in p.nominations
    ).order_by(lambda name, cnt: desc(cnt))[:10]

    df_top = pd.DataFrame(results, columns=['Person', 'Nominations'])
    print("Top 10 most-nominated persons:")
    display(df_top)

top_nominees()

Top 10 most-nominated persons:


,Person,Nominations
0,Metro-Goldwyn-Mayer,68
1,Walt Disney,62
2,John Williams,47
3,Warner Bros.,44
4,France,38
5,Alfred Newman,34
6,Italy,29
7,Paramount,25
8,RKO Radio,24
9,Gordon Hollingshead,23


In [ ]:
# Sample ORM query 2: winners per canonical category 
@db_session
def winners_by_canon():
    results = select(
        (n.category.canon_name, count(n))
        for n in Nomination
        if n.winner
    ).order_by(lambda canon, cnt: desc(cnt))[:10]

    df_w = pd.DataFrame(results, columns=['Canon Category', 'Winners'])
    print("Winners per canonical category (top 10):")
    display(df_w)

winners_by_canon()

Winners per canonical category (top 10):


,Canon Category,Winners
0,HONORARY AWARD,135
1,ACTRESS IN A LEADING ROLE,100
2,ACTOR IN A LEADING ROLE,99
3,WRITING (Adapted Screenplay),97
4,BEST PICTURE,97
5,DIRECTING,96
6,SHORT FILM (Animated),93
7,MUSIC (Original Song),91
8,FILM EDITING,91
9,ACTRESS IN A SUPPORTING ROLE,89


In [ ]:
# Sample ORM query 3: ceremony year range 
@db_session
def ceremony_range():
    years = select(c.year for c in Ceremony)[:]
    print(f"Ceremony year range: {min(years)} to {max(years)}")
    print(f"Total ceremonies: {count(c for c in Ceremony)}")

ceremony_range()

Ceremony year range: 1928 to 2025
Total ceremonies: 97


## Summary

Task 2.1 is complete:

| Item | Detail |
|---|---|
| Database file | `oscar.db` (SQLite 3, via PonyORM) |
| ORM | PonyORM 0.7.19 |
| Entities | `Ceremony`, `Category`, `Person`, `Film`, `Nomination` |
| Rows loaded | 11,110 nominations â†’ 97 ceremonies, 118 categories, 6,893 persons, 5,233 films |
| Nullable relations | `Nomination.person` (7 nulls), `Nomination.film` (359 nulls) |

The database is ready for the Actor Profile App in Task 2.2.